## 4. Validación de Hipótesis de la Cátedra
Carga de librerías y lectura del dataset limpio proporcionado por el equipo.

In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm

df = pd.read_parquet('dataset_limpio.parquet')
df.head()

,product_name,brands,categories,ingredients,nova_group,allergens,energy_kcal,fat_100g,saturated_fat_100g,carbs_100g,...,sodium_100g,contains_gluten,contains_dairy,contains_nuts,contains_soy,contains_eggs,contains_fish,nutriscore_num,ecoscore_num,categoria_macro
0,Sidi Ali,سيدي علي,"en:beverages-and-beverages-preparations, en:be...",OBD1 999 999 1112606 266963207 mb,<NA>,NaN,0.0,0.0,0.0,4.2,...,0.000000,False,False,False,False,False,False,5,<NA>,Bebidas
1,Perly,Perly,"en:dairies, en:fermented-foods, en:fermented-m...","milk cream, cream, sugar, banana, bacteria",3,"en:banana, en:milk",97.0,3.0,0.0,9.4,...,0.000000,False,True,False,False,False,False,<NA>,4,Lácteos
2,اكوافينا,AQUAFINA,"en:beverages-and-beverages-preparations, en:be...",ouverture et avant le : Voir bouteille. après ...,<NA>,NaN,0.0,0.0,0.0,0.0,...,0.000203,False,False,False,False,False,False,5,<NA>,Bebidas
3,Lait,Jaouda,"en:dairies, en:meals, en:milks-liquid-and-powder",Lait frais pasteurisé demi-écrémé à 15g/l de m...,3,en:milk,45.0,1.5,1.0,4.9,...,0.090000,False,True,False,False,False,False,4,2,Lácteos
4,Eau De Source,Cristaline,"en:beverages-and-beverages-preparations, en:be...",Eau de source,1,en:none,0.0,0.0,0.0,0.0,...,0.001100,False,False,False,False,False,False,5,<NA>,Bebidas


### 4.a Proporción de productos altos en grasas saturadas
**Hipótesis:** La proporción de productos catalogados como "altos en grasas saturadas" (más de 5 g de grasa saturada por cada 100 g de producto) en la población de referencia es superior al 25%

* **Variable de interés:** `saturated_fat_100g`
* **Hipótesis Nula ($H_0$):** $p \le 0.25$ (La proporción es menor o igual al 25%)
* **Hipótesis Alternativa ($H_1$):** $p > 0.25$ (La proporción es estrictamente mayor al 25%)
* **Prueba estadística:** Prueba Z para proporciones de una muestra (One-sample proportion Z-test), de cola derecha. Nivel de significancia $\alpha = 0.05$.

In [2]:
grasas_saturadas = df['saturated_fat_100g'].dropna()

altos_en_grasa = grasas_saturadas > 5

exitos = altos_en_grasa.sum()         
n_observaciones = len(grasas_saturadas) 
proporcion_muestral = exitos / n_observaciones

print(f"Productos altos en grasas saturadas: {exitos}")
print(f"Total de observaciones válidas: {n_observaciones}")
print(f"Proporción en la muestra: {proporcion_muestral:.4f} ({proporcion_muestral*100:.2f}%)")

Productos altos en grasas saturadas: 1418
Total de observaciones válidas: 4686
Proporción en la muestra: 0.3026 (30.26%)


### Como el test va a ser aplicado sobre la variable "exitos" (dicotómica), la mejor opcíon en este caso es aplicar un test-Z, ya que solo requiere normalidad (o para nuestro caso un n grande de 4600 observaciones).

In [3]:
from statsmodels.stats.proportion import proportions_ztest

p_0 = 0.25 

z_stat, p_valor = proportions_ztest(count=exitos, nobs=n_observaciones, value=p_0, alternative='larger')

print(f"Estadístico Z: {z_stat:.4f}")
print(f"P-valor: {p_valor:.4e}")

alpha = 0.05
if p_valor < alpha:
    print("\nConclusión: Se rechaza la Hipótesis Nula (H0).")
    print("Hay evidencia estadística para afirmar que la proporción de productos altos en grasas saturadas es superior al 25%.")
else:
    print("\nConclusión: No se rechaza la Hipótesis Nula (H0).")
    print("No hay evidencia estadística suficiente para afirmar que la proporción supera el 25%.")

Estadístico Z: 7.8386
P-valor: 2.2778e-15

Conclusión: Se rechaza la Hipótesis Nula (H0).
Hay evidencia estadística para afirmar que la proporción de productos altos en grasas saturadas es superior al 25%.
